# **Classification Model Training Notebook**



---
## Setup Environment

In [274]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip



You can now save your data files in: /Users/aryan/Machine Learning Assignment 3/36106/assignment/AT3/data


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
utstd 0.1.8 requires scikit-learn~=1.5.1, but you have scikit-learn 1.6.1 which is incompatible.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
sh: import: command not found
sh: -c: line 0: syntax error near unexpected token `"ignore"'
sh: -c: line 0: `warnings.filterwarnings("ignore")'


---
## Student Information

In [275]:
group_name = "36106-26AU-AT3-Group01"
student_name = "Aryan Goel"
student_id = "26040826"

In [276]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [277]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [278]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [279]:
import pandas as pd
import numpy as np
import altair as alt
alt.data_transformers.disable_max_rows()

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression

### 0.b Import Packages

In [280]:
import pandas as pd
import numpy as np

---
## B. Business Understanding

In [281]:
business_use_case_description = """
Business goal
------------
The retail business wants to better understand and anticipate which sales orders are placed through the online channel
vs other channels (e.g., in-store / assisted sales). We will build a classification model that predicts the
online_order_flag for each order using information available at or near order creation time (order attributes and
customer-related attributes, where available).

Why this matters for a retail context
-------------------------------------
Online and offline orders behave differently operationally:
- Online orders typically require pick/pack/ship workflows, last‑mile delivery planning, and more stringent inventory allocation.
- Offline orders are more tied to store foot-traffic, POS staffing, and on-shelf inventory availability.
Accurately predicting the channel (online vs non-online) helps the business plan resources and policies that match the
expected fulfillment path.

Concrete use cases for the prediction
-------------------------------------
1) Fulfillment & capacity planning
   - Forecast the expected volume of online orders for upcoming periods to plan warehouse labor, packing stations,
     courier capacity, and cut-off times.
   - Detect spikes in expected online orders that may require additional shifts or temporary staffing.

2) Inventory allocation and replenishment
   - Decide where to position stock (central warehouse vs store) based on predicted channel demand.
   - Reduce stockouts and “split shipments” by proactively allocating inventory to the right node.

3) Customer experience & service levels
   - Provide more accurate promised delivery dates or pickup windows if an order is likely to be online and shipped.
   - Identify orders likely to require shipping support (address verification, delivery exceptions).

4) Marketing attribution and targeting
   - Segment customers by their likelihood of purchasing online vs offline.
   - Use these segments to tailor campaigns (e.g., free shipping incentives for likely-online customers, store events for
     likely-offline customers).

5) Channel performance monitoring
   - Track how channel mix changes over time and by territory/store coverage.
   - Support strategic decisions such as expanding ecommerce reach in territories with strong predicted online propensity.

Scope of this project
---------------------
- Output: a probability and/or label indicating whether an order is online (online_order_flag).
- Timing: the model should use features that would realistically be known at prediction time (avoid leakage from
  post-fulfillment outcomes).
- Primary users: operations planners, ecommerce managers, supply chain/fulfillment teams, and marketing analytics.
"""

In [282]:
# Do not modify this code
print_tile(size="h3", key='business_use_case_description', value=business_use_case_description)

In [283]:
business_objectives = """
What success looks like
-----------------------
The objective is to produce channel predictions (online vs non-online) that are accurate enough to support
operational and commercial decisions. “Good performance” is not only high overall accuracy, but also strong recall/precision
for the online class if the online channel is the more operationally sensitive one (often it is, due to shipping costs).

Business KPIs impacted (examples)
---------------------------------
1) Fulfillment cost per order
   - Better channel prediction enables more efficient labor scheduling and carrier planning, reducing overtime and
     last-minute carrier premium charges.

2) On-time delivery / service-level adherence (SLA)
   - If online volume is underestimated, the business may miss promised delivery windows due to capacity constraints.
   - If online volume is overestimated, resources may be wasted and offline service may degrade.

3) Inventory availability and stockouts
   - Accurate channel mix improves inventory placement decisions, reducing stockouts and emergency replenishments.

4) Customer satisfaction and retention
   - Late deliveries, cancellations, or poor communication due to mis-planned capacity can hurt customer trust and repeat purchase.

5) Marketing ROI
   - Channel-aware targeting reduces wasted spend (e.g., pushing online promotions to customers who almost always buy offline)
     and improves campaign conversion.

Impact of incorrect predictions (cost of errors)
------------------------------------------------
False Positive (predict online when actually offline):
- May over-allocate warehouse/courier capacity and under-allocate store staffing.
- Could trigger unnecessary shipping-related processes or incentives (e.g., free shipping offers) that reduce margin.

False Negative (predict offline when actually online):
- More risky operationally: insufficient warehouse/courier capacity, delayed fulfillment, missed SLA.
- Increased customer service contacts (WISMO “Where is my order?”), higher refunds/compensation, potential churn.

Risk management and decision thresholds
---------------------------------------
- The business may prefer optimizing for recall of online orders (catch most online orders) even if that increases
  some false positives, because the operational cost of missing online demand can be higher.
- Therefore, we will evaluate more than accuracy: precision/recall/F1 for the “online” class and confusion matrix,
  then select a threshold aligned to business costs.

Constraints / assumptions
-------------------------
- Model must avoid data leakage: features that occur after order placement (e.g., delivery status, returns outcomes)
  should not be used to predict online_order_flag.
- The solution should be explainable enough to be trusted by operations/marketing (e.g., feature importance / reason codes).
"""

In [284]:
# Do not modify this code
print_tile(size="h3", key='business_objectives', value=business_objectives)

In [285]:
stakeholders_expectations_explanations = """
How the results will be used
----------------------------
The model output can be used in two main ways:

1) Real-time / near-real-time operational decisions
   - When an order is created (or during planning runs), score it for likelihood of being online.
   - Route expected online orders into ecommerce fulfillment planning processes (capacity and inventory allocation).

2) Batch analytics and planning
   - Score historical or upcoming orders in bulk to produce forecasts of channel mix by day/week and by territory.
   - Build dashboards that show predicted online share trends and anomalies.

Primary stakeholders and what they need
---------------------------------------
1) Ecommerce / Digital Commerce team
   - Understand expected online demand trends.
   - Measure impact of site changes, promotions, or shipping policies on channel mix.

2) Supply Chain & Fulfillment Operations
   - Staffing and shift planning for warehouse pick/pack/ship.
   - Carrier capacity booking, route planning, and cut-off management.

3) Inventory Planning / Merchandising
   - Decide inventory placement and replenishment rules based on predicted channel demand.
   - Reduce stockouts and improve availability for both online and store channels.

4) Marketing & CRM
   - Identify segments likely to purchase online vs offline for personalization.
   - Improve campaign efficiency and attribution by aligning offers with customer channel preference.

5) Customer Service / Contact Center
   - Prepare for likely increases in shipping-related queries when online volumes rise.
   - Proactively communicate delays if online order volumes exceed capacity.

Who is impacted by the predictions
----------------------------------
- Customers: through delivery reliability, stock availability, and relevance of marketing offers.
- Store staff: through workload planning (in-store demand vs ship-from-store programs).
- Warehouse staff and carriers: through scheduling and volume planning.
- Finance: through margin impacts (shipping costs, promotions, compensation/refunds due to delays).

Expectations for model outputs and reporting
--------------------------------------------
- Provide both:
  (a) a predicted label (online vs non-online), and
  (b) a probability score to support threshold tuning and risk-based decisions.
- Provide model diagnostics for stakeholders:
  - confusion matrix and online-class recall/precision,
  - a short explanation of top drivers (feature importance or coefficients),
  - data quality notes (missingness, coverage differences by territory/store).

Operational adoption expectations
---------------------------------
- The model should be stable over time and monitored (channel behavior can drift due to new promotions, seasonality,
  or changes in distribution/fulfillment).
- The business expects an iterative approach: start with a baseline model, quantify performance, then improve via
  feature engineering, better algorithms, and (if allowed) adding more relevant datasets.
"""

In [286]:
# Do not modify this code
print_tile(size="h3", key='stakeholders_expectations_explanations', value=stakeholders_expectations_explanations)

---
## C. Data Understanding

### C.1   Load Datasets


In [287]:
paths = {
    "customer": at.folder_path / "customer.csv",
    "sales_order_header": at.folder_path / "sales_order_header.csv",
    "sales_order_detail": at.folder_path / "sales_order_detail.csv",
    "product": at.folder_path / "product.csv",
}

dfs = {}
for name, p in paths.items():
    try:
        dfs[name] = pd.read_csv(p)
        print(f"Loaded {name}: {dfs[name].shape}")
    except Exception as e:
        print(f"Could not load {name} ({p.name}): {e}")

if "sales_order_header" in dfs:
    df = dfs["sales_order_header"].copy()
    print("\nUsing df = sales_order_header")
elif "customer" in dfs:
    df = dfs["customer"].copy()
    print("\nUsing df = customer")
else:
    raise FileNotFoundError("No usable dataset found in at.folder_path.")


Loaded customer: (14275, 5)
Loaded sales_order_header: (31465, 17)
Loaded sales_order_detail: (42100, 8)
Loaded product: (886, 23)

Using df = sales_order_header


### C.2 Define Target variable

In [288]:
target_col = None

flag_cols = [c for c in df.columns if ("online" in c.lower() and "flag" in c.lower())]
if flag_cols:
    target_col = flag_cols[0]
elif "status" in df.columns:
    target_col = "status"

if target_col is None:
    raise ValueError(
        "Could not infer a target column. Please choose one from: " + ", ".join(df.columns)
    )

print("Chosen target_col:", target_col)
print("Target sample values:")
display(df[target_col].head(10))


Chosen target_col: online_order_flag
Target sample values:


0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
5    0.0
6    0.0
7    0.0
8    0.0
9    0.0
Name: online_order_flag, dtype: float64

In [289]:
n = len(df)
missing_cnt = df[target_col].isna().sum()
print(f"Rows: {n:,}")
print(f"Missing in target '{target_col}': {missing_cnt:,} ({missing_cnt/n*100:.2f}%)")

Rows: 31,465
Missing in target 'online_order_flag': 0 (0.00%)


In [290]:
# ---------------------------
# C.2 Define Target Variable (extended)
# ---------------------------
import pandas as pd
import numpy as np

# 1) Auto-detect candidate target columns (priority: online flag)
target_col = None
flag_cols = [c for c in df.columns if ("online" in c.lower() and "flag" in c.lower())]

if flag_cols:
    # If multiple, pick the one with the fewest unique values (usually the real flag)
    flag_cols_sorted = sorted(flag_cols, key=lambda c: df[c].nunique(dropna=True))
    target_col = flag_cols_sorted[0]
else:
    # fallback option (only if required / available)
    if "status" in df.columns:
        target_col = "status"

if target_col is None:
    raise ValueError(
        "Could not infer a target column.\n"
        "Hint: create target_col manually, e.g. target_col = 'online_order_flag'\n"
        "Available columns: " + ", ".join(df.columns)
    )

print("Chosen target_col:", target_col)

# 2) Basic target data quality checks
n = len(df)
missing_cnt = df[target_col].isna().sum()
missing_pct = (missing_cnt / n * 100) if n else 0

print(f"Rows: {n:,}")
print(f"Missing in '{target_col}': {missing_cnt:,} ({missing_pct:.2f}%)")

# 3) Show sample raw values (including missing)
print("\nSample values (raw):")
display(df[target_col].head(15))

# 4) Inspect unique labels + their counts
# (Convert to string for a stable view; keep missing separate)
vc = df[target_col].value_counts(dropna=False)
vc_df = vc.reset_index()
vc_df.columns = [target_col, "count"]
vc_df["percent"] = (vc_df["count"] / vc_df["count"].sum() * 100).round(2)

print("\nValue counts (including NaN):")
display(vc_df)

# 5) Determine if the target looks binary
# Normalize labels to string for checking; treat NaN separately
non_null_labels = df[target_col].dropna()

# Helper: map common boolean-ish representations to {0,1}
def normalize_binary_series(s: pd.Series) -> pd.Series:
    s = s.copy()
    # If it's already boolean
    if s.dtype == bool:
        return s.astype(int)

    # If numeric-like, coerce then keep 0/1
    s_num = pd.to_numeric(s, errors="coerce")
    if s_num.notna().all():
        return s_num

    # Otherwise map common strings
    s_str = s.astype(str).str.strip().str.lower()
    mapping = {
        "true": 1, "t": 1, "yes": 1, "y": 1, "1": 1, "online": 1,
        "false": 0, "f": 0, "no": 0, "n": 0, "0": 0, "offline": 0
    }
    return s_str.map(mapping)

normalized = normalize_binary_series(non_null_labels)
unique_norm = pd.Series(normalized.dropna().unique())

looks_binary = False
if len(unique_norm) > 0:
    valid_set = set(unique_norm.tolist())
    looks_binary = valid_set.issubset({0, 1}) and len(valid_set) <= 2

print("\nBinary check:")
print("Looks binary?:", looks_binary)
if looks_binary:
    print("Normalized label set:", sorted(set(unique_norm.tolist())))
else:
    print("Non-binary / multiclass target detected (or contains unexpected labels).")

# 6) Create a clean target series for modeling preview (does NOT modify df unless you assign it)
# Recommended approach:
# - If binary: convert to 0/1 integers (drop rows where it can't be interpreted)
# - If not binary: keep as string classes (drop NaN)
if looks_binary:
    y_clean = normalize_binary_series(df[target_col])
    # Keep only interpretable rows (0/1)
    mask_ok = y_clean.isin([0, 1])
    print(f"\nInterpretable binary rows: {int(mask_ok.sum()):,}/{n:,}")
    print("Rows dropped due to unrecognized labels:", int((~mask_ok).sum()))
    y_clean = y_clean[mask_ok].astype(int)
else:
    y_clean = df[target_col].dropna().astype(str)

print("\nClean target preview:")
display(y_clean.head(20))

# If you want to use it later:
# target_col stays the same (column name)
# y_clean is your cleaned target series

Chosen target_col: online_order_flag
Rows: 31,465
Missing in 'online_order_flag': 0 (0.00%)

Sample values (raw):


0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
Name: online_order_flag, dtype: float64


Value counts (including NaN):


,online_order_flag,count,percent
0,1.0,27659,87.9
1,0.0,3806,12.1



Binary check:
Looks binary?: True
Normalized label set: [0.0, 1.0]

Interpretable binary rows: 31,465/31,465
Rows dropped due to unrecognized labels: 0

Clean target preview:


0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
16    0
17    0
18    0
19    0
Name: online_order_flag, dtype: int64

In [291]:
target_definition_explanations = f"""
C.2 Target definition (detailed)
================================

Target variable: `{target_col}`

1) What the target represents (business meaning)
-----------------------------------------------
`{target_col}` is intended to capture the *order channel* at the time the order is created.
In this project we interpret it as an indicator of whether a sales order was placed through the
online/e‑commerce channel (web / mobile / online checkout) versus a non‑online channel
(e.g., in‑store / assisted sales / sales rep / other offline processes).

In practical retail terms, the target separates two different fulfillment and customer journeys:
- Online orders: typically follow pick/pack/ship workflows, carrier assignment, delivery tracking, and
  potentially higher shipping costs and SLA constraints.
- Non‑online orders: typically rely on store inventory and staff, and may involve immediate fulfillment
  or different operational constraints.

2) Type of machine learning task
--------------------------------
If `{target_col}` is a true online-order flag (two classes), we model this as a **binary classification**
problem with two labels:
- Online (positive class)
- Not online (negative class)

If the column contains values beyond two classes (e.g., multiple channel codes), then the task becomes
**multiclass classification**. For this notebook we assume the intended target is binary; if we discover
more than two distinct valid labels, we would explicitly document them and switch evaluation to multiclass
metrics.

3) Prediction point in time (what information is allowed)
---------------------------------------------------------
The model should predict `{target_col}` using features that are *known at or near order placement time*.
This is important because the model is meant to support planning and operational decisions early in the order lifecycle.

Therefore, we avoid “data leakage” features that are only known after the order is processed, such as:
- delivery outcome / shipping completion timestamps
- return/refund outcomes
- late-payment indicators that occur after checkout
Using leaked features would produce unrealistically high validation scores and a model that fails in real usage.

4) How the target is created / cleaned in this notebook
-------------------------------------------------------
- Missing values:
  - If `{target_col}` has missing values, they are not valid supervised labels.
    We typically **remove those rows** from training (recommended), rather than treating "MISSING" as a class,
    unless the assessment explicitly requires including missing as its own label.
- Data type:
  - We convert labels to a consistent representation (e.g., strings such as "0"/"1" or "False"/"True") so
    the classification pipeline can handle them reliably.
- Class balance:
  - We check the distribution of `{target_col}`. If the online class is rare, accuracy can be misleading.
    We then focus on online-class precision/recall/F1 and may use `class_weight='balanced'` or threshold tuning.

5) Why this target aligns with the business objective
-----------------------------------------------------
Predicting `{target_col}` supports channel-level decision-making because channel mix directly drives:
- operational capacity needs (warehouse vs store)
- inventory allocation strategy (central vs local)
- shipping cost exposure and delivery SLA risk
- marketing personalization and offer strategy (shipping incentives vs store promotions)

In other words, `{target_col}` is a proxy for the operational “path” an order will follow.
Accurately predicting it enables better forecasting of workload and cost, and reduces customer-impacting issues
(e.g., missed delivery windows due to under-planned capacity).

6) Evaluation approach tied to the target
-----------------------------------------
Because misclassifying online orders can be more costly (capacity shortfall, SLA failures), we evaluate:
- confusion matrix to see the error types (false positives vs false negatives)
- precision/recall/F1 for the online class
- (optional) threshold tuning using predicted probabilities, selecting a cutoff that matches the business cost trade-off

This ensures the model is not only statistically accurate but also aligned with real operational risk.
"""

In [292]:
# Do not modify this code
print_tile(size="h3", key='target_definition_explanations', value=target_definition_explanations)

### C.3 Create Target variable

In [293]:
target_name = "y"

df_model = df.copy()

# y as string labels for classification
df_model[target_name] = df_model[target_col].fillna("MISSING").astype(str)

print("Created target column:", target_name)
display(df_model[[target_col, target_name]].head(10))

Created target column: y


,online_order_flag,y
0,0.0,0.0
1,0.0,0.0
2,0.0,0.0
3,0.0,0.0
4,0.0,0.0
5,0.0,0.0
6,0.0,0.0
7,0.0,0.0
8,0.0,0.0
9,0.0,0.0


In [294]:
# ---------------------------
# C.3 Create Target Variable (extended)
# ---------------------------
import pandas as pd
import numpy as np

target_name = "y"
df_model = df.copy()

# 1) Create a cleaned target depending on what the column looks like
s = df_model[target_col]

def make_binary_target(series: pd.Series) -> pd.Series:
    """Try to convert common flag formats to 0/1. Returns float (can contain NaN)."""
    # boolean -> 0/1
    if series.dtype == bool:
        return series.astype(int)

    # numeric-like -> keep
    s_num = pd.to_numeric(series, errors="coerce")
    # If it has at least some numeric and the unique non-null values are subset of {0,1}
    uniq = set(s_num.dropna().unique().tolist())
    if len(uniq) > 0 and uniq.issubset({0, 1}):
        return s_num

    # string mapping
    s_str = series.astype(str).str.strip().str.lower()
    mapping = {
        "true": 1, "t": 1, "yes": 1, "y": 1, "1": 1, "online": 1,
        "false": 0, "f": 0, "no": 0, "n": 0, "0": 0, "offline": 0,
    }
    return s_str.map(mapping)

# Try to interpret as binary flag
y_bin = make_binary_target(s)
uniq_bin = set(y_bin.dropna().unique().tolist())
looks_binary = len(uniq_bin) > 0 and uniq_bin.issubset({0, 1}) and len(uniq_bin) <= 2

print("Target column:", target_col)
print("Looks binary?:", looks_binary)
print("Raw unique (sample):", s.dropna().astype(str).unique()[:10])

# 2) Decide how to store y
# Recommended:
# - If binary, store as 0/1 integers (and drop rows where y is not interpretable)
# - If multiclass, store as string labels (and drop missing)
if looks_binary:
    df_model[target_name] = y_bin
    # Keep only valid 0/1 rows
    before = len(df_model)
    df_model = df_model[df_model[target_name].isin([0, 1])].copy()
    df_model[target_name] = df_model[target_name].astype(int)
    after = len(df_model)
    print(f"Dropped rows with invalid/unrecognized target labels: {before-after}")
else:
    # Multiclass / categorical: keep strings, but drop missing
    df_model[target_name] = df_model[target_col].astype("string")
    before = len(df_model)
    df_model = df_model.dropna(subset=[target_name]).copy()
    df_model[target_name] = df_model[target_name].astype(str).str.strip()
    after = len(df_model)
    print(f"Dropped rows with missing target: {before-after}")

# 3) Final checks and display
print("\nCreated target column:", target_name)
print("df_model shape:", df_model.shape)
print("Target value counts:")
display(df_model[target_name].value_counts(dropna=False).to_frame("count"))

print("\nTarget preview:")
display(df_model[[target_col, target_name]].head(10))

# 4) Optional: if you want a human-readable label for binary targets
if looks_binary:
    df_model["y_label"] = df_model[target_name].map({0: "not_online", 1: "online"})
    print("\nAdded y_label (optional). Example counts:")
    display(df_model["y_label"].value_counts().to_frame("count"))

# After this cell:
# - use df_model as your modeling dataframe
# - y is in df_model["y"]

Target column: online_order_flag
Looks binary?: True
Raw unique (sample): ['0.0' '1.0']
Dropped rows with invalid/unrecognized target labels: 0

Created target column: y
df_model shape: (31465, 18)
Target value counts:


,count
y,
1,27659
0,3806



Target preview:


,online_order_flag,y
0,0.0,0
1,0.0,0
2,0.0,0
3,0.0,0
4,0.0,0
5,0.0,0
6,0.0,0
7,0.0,0
8,0.0,0
9,0.0,0



Added y_label (optional). Example counts:


,count
y_label,
online,27659
not_online,3806


### C.4 Explore Target variable

In [295]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# Ensure y exists
target_name = "y"
if target_name not in df_model.columns:
    raise ValueError("Run C3 first to create df_model['y'].")

# 1) Correct distribution table
dist = (
    df_model[target_name]
    .value_counts(dropna=False)
    .rename_axis("y")              # name the index properly
    .reset_index(name="count")     # now we have columns: y, count
)
dist["percent"] = (dist["count"] / dist["count"].sum() * 100).round(2)

# Add readable labels
dist["y_label"] = dist["y"].map({0: "not_online", 1: "online"}).fillna(dist["y"].astype(str))

print("Target distribution (correct):")
display(dist)

# 2) Counts chart
counts_chart = alt.Chart(dist).mark_bar().encode(
    x=alt.X("y_label:N", sort="-y", title="Target class"),
    y=alt.Y("count:Q", title="Count"),
    tooltip=[
        alt.Tooltip("y_label:N", title="Class"),
        alt.Tooltip("count:Q", title="Count", format=",.0f"),
        alt.Tooltip("percent:Q", title="Percent", format=".2f"),
    ],
    color=alt.Color("y_label:N", legend=None)
).properties(width=650, height=300, title="C4: Target distribution (counts)")

# 3) Percent chart
percent_chart = alt.Chart(dist).mark_bar().encode(
    x=alt.X("y_label:N", sort="-y", title="Target class"),
    y=alt.Y("percent:Q", title="Percent (%)"),
    tooltip=[
        alt.Tooltip("y_label:N", title="Class"),
        alt.Tooltip("percent:Q", title="Percent", format=".2f"),
        alt.Tooltip("count:Q", title="Count", format=",.0f"),
    ],
    color=alt.Color("y_label:N", legend=None)
).properties(width=650, height=300, title="C4: Target distribution (%)")

# 4) Territory chart (stacked) - top N only
if "territory_id" in df_model.columns:
    agg = (
        df_model[["territory_id", target_name]]
        .assign(
            territory_id=lambda d: d["territory_id"].astype(str),
            y_label=lambda d: d[target_name].map({0: "not_online", 1: "online"}).astype(str)
        )
        .groupby(["territory_id", "y_label"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
    )

    top_n = 12
    top_terr = (
        agg.groupby("territory_id")["count"].sum()
           .sort_values(ascending=False).head(top_n).index
    )
    agg_top = agg[agg["territory_id"].isin(top_terr)].copy()
    agg_top["territory_short"] = agg_top["territory_id"].str.slice(0, 8)

    terr_chart = alt.Chart(agg_top).mark_bar().encode(
        y=alt.Y("territory_short:N", sort="-x", title=f"Territory (top {top_n})"),
        x=alt.X("count:Q", title="Orders"),
        color=alt.Color("y_label:N", title="Target (y)"),
        tooltip=[
            alt.Tooltip("territory_id:N", title="Territory ID"),
            alt.Tooltip("y_label:N", title="Class"),
            alt.Tooltip("count:Q", title="Count", format=",.0f"),
        ],
    ).properties(width=750, height=380, title="C4: Target counts by territory (stacked)")
else:
    terr_chart = alt.Chart(pd.DataFrame({"note": ["territory_id not in df_model"]})).mark_text().encode(text="note:N")

# Show charts
(counts_chart & percent_chart & terr_chart).configure(background="white").configure_view(stroke=None)

Target distribution (correct):


,y,count,percent,y_label
0,1,27659,87.9,online
1,0,3806,12.1,not_online


alt.VConcatChart(...)

In [296]:
display(
    df_model[[target_col, "y"]]
    .dropna()
    .astype({target_col: "string"})
    .head(20)
)

print("Unique values in original target_col:", df_model[target_col].dropna().unique()[:10])
print("Unique values in y:", sorted(df_model["y"].unique()))
print(df_model["y"].value_counts())

,online_order_flag,y
0,0.0,0
1,0.0,0
2,0.0,0
3,0.0,0
4,0.0,0
5,0.0,0
6,0.0,0
7,0.0,0
8,0.0,0
9,0.0,0


Unique values in original target_col: [0. 1.]
Unique values in y: [np.int64(0), np.int64(1)]
y
1    27659
0     3806
Name: count, dtype: int64


In [ ]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# --- Build agg table safely ---
if "territory_id" not in df_model.columns:
    print("territory_id not found in df_model")
else:
    by_terr = df_model[["territory_id", target_name]].copy()

    # normalize types (important)
    by_terr["territory_id"] = by_terr["territory_id"].fillna("MISSING").astype(str)
    by_terr[target_name] = by_terr[target_name].fillna("MISSING").astype(str)

    agg = (
        by_terr
        .groupby(["territory_id", target_name], dropna=False)
        .size()
        .reset_index(name="count")
    )

    print("agg shape:", agg.shape)
    display(agg.head(10))

    # If agg is empty, stop early with a clear message
    if agg.empty:
        raise ValueError("Aggregation produced 0 rows. df_model may be empty or columns missing.")

    # --- Keep only top territories to make the chart readable ---
    top_n = 15
    top_territories = (
        agg.groupby("territory_id")["count"]
           .sum()
           .sort_values(ascending=False)
           .head(top_n)
           .index
           .tolist()
    )

    agg_top = agg[agg["territory_id"].isin(top_territories)].copy()

    # --- Plot (STACKED) ---
    chart = (
        alt.Chart(agg_top)
        .mark_bar()
        .encode(
            y=alt.Y("territory_id:N", sort="-x", title="Territory (top 15)"),
            x=alt.X("count:Q", title="Orders"),
            color=alt.Color(f"{target_name}:N", title=target_name),
            tooltip=[
                alt.Tooltip("territory_id:N", title="Territory"),
                alt.Tooltip(f"{target_name}:N", title=target_name),
                alt.Tooltip("count:Q", title="Count", format=",.0f"),
            ],
        )
        .properties(width=750, height=380, title=f"C4: {target_name} counts by territory (top {top_n})")
        .configure(background="white")
        .configure_view(stroke=None)
    )

    chart

agg shape: (20, 3)


,territory_id,y,count
0,0ad4c625-bb65-4376-8a41-0a65719b0db8,0,188
1,0ad4c625-bb65-4376-8a41-0a65719b0db8,1,2484
2,1e179cbb-7db9-4a66-aa94-e6fb9b3d613e,0,469
3,1e179cbb-7db9-4a66-aa94-e6fb9b3d613e,1,17
4,25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,0,139
5,25d73ad2-9e13-410b-8b0d-5f26e2b9e4f2,1,2484
6,2ac923e9-3043-4f5e-bae0-ad28089bf187,0,125
7,2ac923e9-3043-4f5e-bae0-ad28089bf187,1,6718
8,5b963d6f-3cda-4843-ade2-0cb22c0eccaf,0,376
9,5b963d6f-3cda-4843-ade2-0cb22c0eccaf,1,9


In [ ]:
print("df_model shape:", df_model.shape)
print("territory_id missing:", df_model["territory_id"].isna().sum() if "territory_id" in df_model.columns else "N/A")
print("y missing:", df_model[target_name].isna().sum() if target_name in df_model.columns else "N/A")
display(df_model[["territory_id", target_name]].head(10))


df_model shape: (31465, 19)
territory_id missing: 0
y missing: 0


,territory_id,y
0,1e179cbb-7db9-4a66-aa94-e6fb9b3d613e,0
1,1e179cbb-7db9-4a66-aa94-e6fb9b3d613e,0
2,c8b914d0-8c4c-498f-af9f-6954c90f45db,0
3,c8b914d0-8c4c-498f-af9f-6954c90f45db,0
4,5f568b38-d738-4b7a-a27b-bc975b9084a2,0
5,d90fce1e-44e0-4d8f-8125-2f9ba1d18cbc,0
6,d90fce1e-44e0-4d8f-8125-2f9ba1d18cbc,0
7,5f568b38-d738-4b7a-a27b-bc975b9084a2,0
8,5b963d6f-3cda-4843-ade2-0cb22c0eccaf,0
9,c8b914d0-8c4c-498f-af9f-6954c90f45db,0


In [298]:
target_distribution_explanations = """
Target variable definition
--------------------------
The target variable `y` is created directly from `online_order_flag`, which indicates the channel through which the
sales order was placed:
- y = 1  → Online order (e-commerce channel)
- y = 0  → Not online / offline order (e.g., in-store or assisted channel)

Observed distribution (sales_order_header)
------------------------------------------
Total rows: 31,465
Missing target values: 0 (0.00%)

Class counts and proportions:
- Online (y=1): 27,659 orders (87.9%)
- Offline (y=0): 3,806 orders (12.1%)

This distribution shows a clear class imbalance, where online orders dominate the dataset. The ratio is approximately
7.27:1 (online:offline). In operational terms, the dataset reflects that most recorded orders in this table occur
through the online channel, while a smaller share occurs through non-online channels.

Why this matters (limitations and risks)
----------------------------------------
1) Accuracy can be misleading:
   If a model predicts every order as online (y=1), it would already achieve ~87.9% accuracy without learning any real
   pattern. Therefore, accuracy alone is not a reliable indicator of model usefulness.

2) Minority-class performance is critical:
   Offline orders (y=0) are the minority class (12.1%). If the model performs poorly on this class, it may fail to detect
   important offline demand signals. This can cause biased decisions such as under-allocating store staffing or
   misinterpreting channel performance.

3) Metric selection must reflect imbalance:
   We need to report class-sensitive metrics, especially for the minority class:
   - Precision (offline): when the model predicts offline, how often is it correct?
   - Recall (offline): how many offline orders are correctly detected?
   - F1-score (offline): balances precision and recall to avoid optimizing one at the expense of the other
   Additionally, the confusion matrix is essential to understand the types of mistakes (false positives vs false negatives).

4) Modeling and threshold considerations:
   Because of imbalance, we may need to:
   - use `class_weight='balanced'` (or similar weighting) so the learning algorithm penalizes mistakes on offline orders more
   - tune the probability threshold (instead of using default 0.5) to match business costs (e.g., if missing offline orders
     is more harmful than falsely flagging some online orders as offline)

5) Validation strategy:
   We should use stratified splits (train/validation/test) so that each split keeps a similar 87.9/12.1 distribution.
   Otherwise, performance estimates can be unstable and not representative.

Conclusion
----------
The target distribution is suitable for binary classification, but the strong imbalance means the model evaluation must
focus on minority-class performance and cost-sensitive decision-making rather than accuracy alone.
"""

In [299]:
# Do not modify this code
print_tile(size="h3", key='target_distribution_explanations', value=target_distribution_explanations)

### C.5 Explore Feature of Interest `\<put feature name here\>`

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
feature_1_insights = """
provide a detailed analysis on the selected feature, its distribution, limitations, issues, ...
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### C.6 Explore Feature of Interest `\<put feature name here\>`

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
feature_2_insights = """
provide a detailed analysis on the selected feature, its distribution, limitations, issues, ...
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### C.6 Explore Feature of Interest `\<put feature name here\>`


In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
feature_n_insights = """
provide a detailed analysis on the selected feature, its distribution, limitations, issues, ...
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

### C.n Explore Feature of Interest `\<put feature name here\>`

> You can add more cells related to other feeatures in this section

---
## D. Feature Selection


In [ ]:
# <Student to fill this section>

features_list = []

In [ ]:
# <Student to fill this section>
feature_selection_explanations = """
provide a quick explanation on the features selected
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## E. Data Preparation

### E.1 Data Transformation <put_name_here>

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
data_cleaning_1_explanations = """
Provide some explanations on why you believe it is important to fix this issue and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### E.2 Data Transformation <put_name_here>

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
data_cleaning_2_explanations = """
Provide some explanations on why you believe it is important to fix this issue and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### E.3 Data Transformation <put_name_here>

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
data_cleaning_3_explanations = """
Provide some explanations on why you believe it is important to fix this issue and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

### E.n Fixing "\<describe_issue_here\>"

> You can add more cells related to other issues in this section

---
## F. Feature Engineering

### F.1 New Feature "\<put_name_here\>"


In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
feature_engineering_1_explanations = """
Provide some explanations on why you believe it is important to create this feature and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### F.2 New Feature "\<put_name_here\>"




In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
feature_engineering_2_explanations = """
Provide some explanations on why you believe it is important to create this feature and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### F.3 New Feature "\<put_name_here\>"

> Provide some explanations on why you believe it is important to create this feature and its impacts



In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
feature_engineering_n_explanations = """
Provide some explanations on why you believe it is important to create this feature and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

### F.n Fixing "\<describe_issue_here\>"

> You can add more cells related to new features in this section

---
## G. Data Preparation for Modeling

### G.1 Split Datasets

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
data_splitting_explanations = """
Provide some explanations on what is the best strategy to use for data splitting for this dataset
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_splitting_explanations', value=data_splitting_explanations)

### G.2 Data Transformation "\<put_name_here\>"

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
data_transformation_1_explanations = """
Provide some explanations on why you believe it is important to perform this data transformation and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### G.3 Data Transformation "\<put_name_here\>"

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
data_transformation_2_explanations = """
Provide some explanations on why you believe it is important to perform this data transformation and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

### G.4 Data Transformation "\<put_name_here\>"

In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
data_transformation_3_explanations = """
Provide some explanations on why you believe it is important to perform this data transformation and its impacts
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='data_transformation_3_explanations', value=data_transformation_3_explanations)

---
## H. Save Datasets

> Do not change this code

In [ ]:
# Do not modify this code
# Save training set
try:
  X_train.to_csv(at.folder_path / 'X_train.csv', index=False)
  y_train.to_csv(at.folder_path / 'y_train.csv', index=False)

  X_val.to_csv(at.folder_path / 'X_val.csv', index=False)
  y_val.to_csv(at.folder_path / 'y_val.csv', index=False)

  X_test.to_csv(at.folder_path / 'X_test.csv', index=False)
  y_test.to_csv(at.folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)

name 'X_train' is not defined


## J. Train Machine Learning Model

### J.1 Import Algorithm

> Provide some explanations on why you believe this algorithm is a good fit


In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
algorithm_selection_explanations = """
Provide some explanations on why you believe this algorithm is a good fit
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### J.2 Set Hyperparameters

> Provide some explanations on why you believe this algorithm is a good fit


In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
hyperparameters_selection_explanations = """
Explain why you are tuning these hyperparameters
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### J.3 Fit Model

In [ ]:
# <Student to fill this section>

### J.4 Model Technical Performance

> Provide some explanations on model performance


In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
model_performance_explanations = """
Provide some explanations on model performance
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### J.5 Business Impact from Current Model Performance

> Provide some analysis on the model impacts from the business point of view


In [ ]:
# <Student to fill this section>

In [ ]:
# <Student to fill this section>
business_impacts_explanations = """
Interpret the results of the experiments related to the business objective set earlier. Estimate the impacts of the incorrect results for the business (some results may have more impact compared to others)
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## H. Project Outcomes

In [ ]:
# <Student to fill this section>
experiment_outcome = "" # Either 'Hypothesis Confirmed', 'Hypothesis Partially Confirmed' or 'Hypothesis Rejected'

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [ ]:
# <Student to fill this section>
experiment_results_explanations = """
Reflect on the outcome of the experiment and list the new insights you gained from it. Provide rationale for pursuing more experimentation with the current approach or call out if you think it is a dead end.
Given the results achieved and the overall objective of the project, list the potential next steps and experiments. For each of them assess the expected uplift or gains and rank them accordingly. If the experiment achieved the required outcome for the business, recommend the steps to deploy this solution into production.
"""

In [ ]:
# Do not modify this code
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)